# Assignment 2 - RAG
Due date: 28.4.26
Students: Yahel Cohen, Gabriel Abramovich

Intro
In this assignment you’ll familiarize yourself with RAG - Retrieval Augmented Generation,
first in theory (just a bit) and then we’ll dive into practicalities. You’ll start querying without
RAG and after adding the RAG component you’ll compare the outcomes. The last steps will
focus on RAG evaluation metrics and improvement cycles.
By the end of this assignment you'll have built a working RAG pipeline, evaluated it across
three different dimensions (not just final-answer correctness), and run improvement cycles
on it. The goal isn't to hit a particular accuracy number - financebench is hard on purpose -
but to develop intuition for which component to fix when results disappoint, which is the skill
that matters in production.

General Instructions
1. Submit one notebook that contains the code and textual answers for all tasks.
2. Some tasks also require xlsx files - pay attention.
3. Zip it all together and include first and last names of both students in the file name.
4. Note: I usually don’t run the notebook, but there should be placeholders for the data
loading, api_key etc. in case I do need to run it (it happens. Rarely, but it happens).
5. Having said that, you need to run it of course, and outputs should be present.

Dataset
We’ll use the financebench dataset. Read the description and read carefully through the
columns and make sure you understand them (you can ignore dataset_subset_label). The
paper can also help you with that.
Notes:
1. The dataset has 3 types of questions: metrics-generated, domain-relevant,
novel-generated. Drop the metrics-generated questions.
2. Some of the urls in the doc_link column are dead. Replace them with links to this
repo. Note that the folder contains more documents than are referenced by the
dataset.

## Task 1 - naive generation
Use the Llama-3.3-70B-Instruct model (via Nebius Token Factory) to answer the first 5
questions (simply sort by financebench_id) of each question_type - 5 domain-relevant, 5
novel-generated.
Note: No retrieval, just the question straight into the model.
For each question, compare the model's output with the expected answer (aka "ground truth"
or "gold answer").
Note: the model might not provide an answer and instead ask for more information, refuse,
or hedge. That's itself a result worth recording.
Look at the answers and identify:
1. Cases where the model refused or asked for more information - why?
2. Cases where the model answered confidently - spot-check against the ground
truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
3. Are there patterns by question_type? Do some types fail more than others?
Deliverables:
1. The code.
2. A table (xlsx) with columns: financebench_id | question_type | question |
naive_answer | ground_truth | verdict, where verdict is one of {correct, partially
correct, wrong, refused} based on your manual judgment.
Name the file assignment2_naive_generation.
3. A short written discussion (markdown cell in the notebook) addressing the three
questions above.

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import time     
import pandas as pd

load_dotenv()

True

In [5]:
# ---- Task 1: Naive generation (no retrieval) ----
from datasets import load_dataset

# 1. Load FinanceBench, drop metrics-generated, take first 5 of each remaining type
ds = load_dataset("PatronusAI/financebench", split="train").to_pandas()
ds = ds[ds["question_type"] != "metrics-generated"].copy()
sample = (
    ds.sort_values("financebench_id")
      .groupby("question_type", group_keys=False)
      .head(5)
      .reset_index(drop=True)
)
print(f"Sampled {len(sample)} questions: {sample['question_type'].value_counts().to_dict()}")

# 2. Nebius Token Factory client (OpenAI-compatible)
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv("NEBIUS_API_KEY"),
)
MODEL = "meta-llama/Llama-3.3-70B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial analyst assistant. Answer the user's question as accurately "
    "as you can. If you do not have enough information to answer, say so clearly "
    "instead of guessing."
)

def ask_naive(question: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    return resp.choices[0].message.content.strip()

Sampled 10 questions: {'domain-relevant': 5, 'novel-generated': 5}


In [ ]:
# 3. Run inference
answers = []
for i, row in sample.iterrows():
    print(f"[{i+1}/{len(sample)}] {row['financebench_id']} ({row['question_type']})")
    try:
        answers.append(ask_naive(row["question"]))
    except Exception as e:
        answers.append(f"ERROR: {e}")
    time.sleep(0.4)  # gentle rate-limit

sample["naive_answer"] = answers

# 4. Build deliverable table — verdict left blank for manual judgment
out = (
    sample[["financebench_id", "question_type", "question", "naive_answer", "answer"]]
      .rename(columns={"answer": "ground_truth"})
)
out["verdict"] = ""  # fill manually with: correct | partially correct | wrong | refused

out.to_excel("assignment2_naive_generation.xlsx", index=False)
out

README.md: 0.00B [00:00, ?B/s]

financebench_merged.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

Sampled 10 questions: {'domain-relevant': 5, 'novel-generated': 5}
[1/10] financebench_id_00005 (domain-relevant)
[2/10] financebench_id_00070 (domain-relevant)
[3/10] financebench_id_00080 (domain-relevant)
[4/10] financebench_id_00206 (domain-relevant)
[5/10] financebench_id_00215 (domain-relevant)
[6/10] financebench_id_00283 (novel-generated)
[7/10] financebench_id_00288 (novel-generated)
[8/10] financebench_id_00299 (novel-generated)
[9/10] financebench_id_00302 (novel-generated)
[10/10] financebench_id_00382 (novel-generated)


,financebench_id,question_type,question,naive_answer,ground_truth,verdict
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,Yes. Corning had a positive working capital am...,
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,To determine whether American Water Works has ...,"No, American Water Works had negative working ...",
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,To determine if PayPal has positive working ca...,Yes. Paypal has a positive working capital of ...,
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"To answer this question, I'll need to look at ...","Since JPM is a financial institution, gross ma...",
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,To determine if Verizon is a capital-intensive...,Yes. Verizon's capital intensity ratio was app...,
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,Pfizer announced plans to spin off its Upjohn ...,77.78,
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer your...,"Yes, there was a decline of ~42% between FY202...",
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,According to JPMorgan Chase's Q1 2021 earnings...,Corporate. Its net revenue was -$473 million.,
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,To answer whether Pfizer grew its Pre-tax Prof...,"Yes, change in PPNE was positive year over year",
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have access to real-time data or MGM's...,Las Vegas resorts contributed ~90% of company ...,


Questions:
1. Cases where the model refused or asked for more information - why?
2. Cases where the model answered confidently - spot-check against the ground truth. Is the answer correct? Partially correct? Totally wrong (hallucinated)?
3. Are there patterns by question_type? Do some types fail more than others?

Answers:
1. The cases where the model refused or asked for more information could correlate with information the model could not have memorized like recent quarters, segment level breakdowns.
2. Looking at the generations, there is not a single correct answer which indicates the model abillity to learn things up to a certain granularity. However, the model do had 4 partially-correct answers where he did got the general verdict right (yes/no, capital-intensive/not, etc.) but hallucinates the supporting data used to justify it. He also had 2 cases where he totally fabricated the answer.
3. Yes, there are some patterns that the question_type yields. The domain-relevant are more high-level, so the model hallucinate more than the novel-generated questions where the questions are about narrower slices.

## Task 2 - RAG reminder (5 points)  
Below is a sketch of a simple RAG pipeline. The boxes group into three components - 
indexing (documents → chunk + embed → vector store), retrieval (user query → retrieval), 
and generation. For each component, briefly explain:  
1. How does it contribute to the pipeline?  
2. Where can it fail? Try to think of concrete examples.  
3. Does it happen once ("offline"), or per query?  
Deliverable: A short write-up (markdown cell in the notebook), ~2-4 sentences per 
component covering the three questions above. 
 

Answers:
1. Indexing is the way that we store and prepare the data in a semantic way, so later we can query the actual meaning of the data.
It can fail on couple of things such as wrong chuncking, that can lead to too large/small documents or meaningful data that gets cut to pieces, wrong indexing of multiple languages or even weak embeding model. The process is happened once, offline.
2. Retrieval is the process where we go to the actual data that we store and getting the relevant documents that matches theuser intent, by his query. Because the retrieval is the heart of the pipeline getting the wrong documents will yields the wrong outcome, wrong ranking order for example, the right document is ranked 5 or simply returning too many docs. This step is happening every time the user query the model.
3. Generation is the purpose of the pipeline. Eventually, this is the actual answer that the user expects. It can fail on empty retreival which can lead to hallucinations, not strict enough prompt and instructions. It happens online on every user query.

## Task 3 - embed documents (15 points)  
The source documents (linked again here) are going to be the knowledge base for your 
vector store.  
Instructions:  
1. Load PDFs with PyPDFLoader (one Document per page). Recall that not all pdfs are 
relevant, use only the PDFs corresponding to doc_name values that appear in your 
filtered dataset.  
2.  Attach  metadata  to  each  page  before  splitting:  doc_name,  company,  doc_period, 
page_number.  Keep  page_number  0-indexed  to  match  the  dataset's 
evidence_page_num.  
Note: PyPDFLoader already attaches a default page metadata field, but we're 
standardizing on page_number for clarity.  
3. Split with RecursiveCharacterTextSplitter, chunk_size=1000, chunk_overlap=150. 
Chunks inherit page metadata.  
4. Store in a LangChain FAISS vector store using BAAI/bge-small-en-v1.5 from 
Hugging-Face as the embedding model.  
5. Save the FAISS index to disk (vectorstore.save_local(...)) so you don't re-embed every 
time you restart your notebook. You'll reuse this index across Tasks 4–7.  

Pick 2-3 questions from the dataset and retrieve the top-k chunks for each. For each 
retrieval, check:  
- Did the retrieved chunks come from the right document (matching doc_name)? 
- Do they contain (or come close to) the evidence text from the dataset's evidence field?  
- Do they come from the right page (chunk page_number vs evidence_page_num)? 
Hint: look again at these results when working on task 7.  
Deliverables:  
1. The code (loading, metadata, splitting, embedding, saving).  
2. A short markdown cell with your observations.

In [9]:
# ---- Task 3.0: install the libs we need (safe to re-run; cached after first install) ----
# `cryptography` is needed by pypdf to read AES-encrypted PDFs (e.g. ADOBE_2022_10K)
!pip3 install --quiet langchain-community langchain-huggingface langchain-text-splitters faiss-cpu pypdf cryptography sentence-transformers



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [10]:
# ---- Task 3.1: collect doc_names + helper to fetch PDFs on the fly ----
import urllib.request
from io import BytesIO

# `ds` was filtered above (metrics-generated dropped) → only pull the PDFs we need
doc_names = sorted(ds["doc_name"].unique())
print(f"Filtered dataset references {len(doc_names)} unique PDFs")

# financebench raw PDFs live at: pdfs/{doc_name}.pdf in the patronus-ai/financebench repo
RAW_BASE = "https://raw.githubusercontent.com/patronus-ai/financebench/main/pdfs"

def fetch_pdf_bytes(name: str) -> bytes:
    """Download a single PDF on the fly and return its bytes — no disk I/O."""
    with urllib.request.urlopen(f"{RAW_BASE}/{name}.pdf") as resp:
        return resp.read()

# doc_name → company / doc_period (taken from the filtered dataset itself)
meta_lookup = (
    ds.drop_duplicates("doc_name")
      .set_index("doc_name")[["company", "doc_period"]]
      .to_dict("index")
)


Filtered dataset references 42 unique PDFs


In [11]:
# ---- Task 3.2: parse PDFs in-memory + attach metadata ----
# Note: the assignment suggests PyPDFLoader, but it requires a file path on disk.
# To keep everything in RAM (no local PDF cache) we use pypdf directly — same
# parser PyPDFLoader wraps, same one-Document-per-page result, fed BytesIO
# instead of a path.
from pypdf import PdfReader
from langchain_core.documents import Document

pages, failed = [], []
for name in doc_names:
    try:
        reader = PdfReader(BytesIO(fetch_pdf_bytes(name)))
    except Exception as e:
        failed.append((name, str(e)))
        continue
    info = meta_lookup.get(name, {})
    for page_idx, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(Document(
            page_content=text,
            metadata={
                "doc_name": name,
                "company": info.get("company"),
                "doc_period": info.get("doc_period"),
                "page_number": page_idx,  # 0-indexed, matches evidence_page_num
            },
        ))

print(f"Loaded {len(pages)} pages across {len(doc_names) - len(failed)} PDFs (failed: {len(failed)})")
print("Sample metadata:", pages[0].metadata)
for n, e in failed[:5]:
    print(" - failed:", n, "->", e)


Loaded 5349 pages across 41 PDFs (failed: 1)
Sample metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}
 - failed: ADOBE_2022_10K -> cryptography>=3.1 is required for AES algorithm


In [12]:
# ---- Task 3.3: split into chunks (chunks inherit page metadata) ----
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages)

print(f"{len(pages)} pages → {len(chunks)} chunks")
print("Sample chunk metadata:", chunks[0].metadata)
print("Sample chunk text (first 240 chars):")
print(chunks[0].page_content[:240])


5349 pages → 23740 chunks
Sample chunk metadata: {'doc_name': '3M_2022_10K', 'company': '3M', 'doc_period': 2022, 'page_number': 0}
Sample chunk text (first 240 chars):
T able of Contents
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 
10-K
 
 
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
 
December 31
, 2022
or


In [13]:
# ---- Task 3.4: embed chunks + build / load FAISS index ----
from langchain_community.vectorstores import FAISS
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

embedder = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},  # bge expects normalized vectors for cosine
)

INDEX_DIR = "faiss_index_chunk1000"

if Path(INDEX_DIR).exists():
    vectorstore = FAISS.load_local(
        INDEX_DIR, embedder, allow_dangerous_deserialization=True
    )
    print(f"Loaded existing FAISS index from {INDEX_DIR}/  (skipped re-embedding)")
else:
    vectorstore = FAISS.from_documents(chunks, embedder)
    vectorstore.save_local(INDEX_DIR)
    print(f"Built and saved FAISS index → {INDEX_DIR}/")

print(f"Index size: {vectorstore.index.ntotal} vectors")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Built and saved FAISS index → faiss_index_chunk1000/
Index size: 23740 vectors


In [14]:
# ---- Task 3.5: sanity-check retrieval on a few questions ----
# Pick 3 questions spanning both question_types from the Task 1 sample
probe_ids = ["financebench_id_00005", "financebench_id_00283", "financebench_id_00382"]
probes = ds[ds["financebench_id"].isin(probe_ids)].copy()

for _, row in probes.iterrows():
    q = row["question"]
    gold_doc = row["doc_name"]
    # `evidence` is a list of dicts with evidence_text + evidence_page_num
    evidence = row["evidence"] if row["evidence"] is not None else []
    gold_pages = [e["evidence_page_num"] for e in evidence]
    gold_text = " | ".join(e["evidence_text"][:140] for e in evidence)

    print("=" * 100)
    print(f"[{row['financebench_id']}] {q}")
    print(f"  gold doc:   {gold_doc}")
    print(f"  gold pages: {gold_pages}")
    print(f"  gold evidence (truncated): {gold_text[:240]}")

    hits = vectorstore.similarity_search(q)
    for i, h in enumerate(hits, 1):
        m = h.metadata
        same_doc  = m["doc_name"] == gold_doc
        same_page = m["page_number"] in gold_pages
        print(f"  rank {i}: doc={m['doc_name']} page={m['page_number']} "
              f"(doc✓={same_doc}, page✓={same_page})")
        snippet = h.page_content[:180].replace("\n", " ")
        print(f"    > {snippet}")
    print()


[financebench_id_00005] Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.
  gold doc:   CORNING_2022_10K
  gold pages: [59]
  gold evidence (truncated): Consolidated Balance Sheets
Corning Incorporated and Subsidiary Companies
 
 
 
December 31,
 
(in millions, except share and per share amou
  rank 1: doc=CORNING_2022_10K page=101 (doc✓=True, page✓=False)
    > (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020. 
  rank 2: doc=CORNING_2022_10K page=102 (doc✓=True, page✓=False)
    > (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Growth Businesses since September 9, 2020. 
  rank 3: doc=3M_2022_10K page=37 (doc✓=False, page✓=False)
    > not defin

Questions:
- Did the retrieved chunks come from the right document (matching doc_name)? 
- Do they contain (or come close to) the evidence text from the dataset's evidence field?  
- Do they come from the right page (chunk page_number vs evidence_page_num)?

Answer:

Only the Corning retrival was containing chunks from the right document but not the correct evidence_page_num. Pfizer & MGM cases were actually close by meaning of the chunks, for Pfizer the question was: "How much does Pfizer expect to pay to spin off Upjohn in the future in USD million?". the gold evidence is: "We expect to incur costs of approximately $700 million in connection with separating Upjohn, of which approximately 90% has been incurred si" and the top retrieved chunk was: "to the UpjohnBusiness, with the remainder considered a non-cash activity in the consolidated statement of cash flows for the year ended December 31, 2020. The spin-off also resulte". both contain Upjohn text, both are about Pfizer. The chunks themselves don't carry filing identity in their text. The embedding has no signal to prefer Pfizer_2023Q2_10Q (the right one) over PFIZER_2021_10K.

## Task 4 - time for some RAG! let’s build a RAG pipeline (25 points)
Write code that takes a user query, retrieves the most relevant chunks from the vector store,
and feeds them - along with the query - into the generation model to produce a final answer.
Use the same embedding model from Task 3 (BAAI/bge-small-en-v1.5) to embed the user’s
query, and the same generation model from Task 1(Llama-3.3-70B-Instruct) to produce the
final answer.

Prompt construction. Think about how you format the retrieved chunks in the prompt: use
clear separators between chunks, and include the doc_name metadata so the model knows
which filing each chunk came from. Handle the case where retrieval returns an empty
retrieval - the model should be told there's no relevant context rather than being handed an
empty block.
System prompt. Write a system prompt that instructs the model to:
- Answer only from the provided context.
- Say explicitly when the context doesn't contain the answer, rather than guessing.
- Keep answers concise and cite the document each fact came from.

Deliverables:
1. The code. Make sure to include the prompts.
2. Wrap everything in a single function:
answer_with_rag(query: str, k: int = 4) -> dict
The returned dict should contain:
- answer (str): the generation model's final answer.
- retrieved_chunks (list): the chunks used as context, each with its
doc_name and page_number metadata.

In [15]:
# ---- Task 4: RAG pipeline ----

SYSTEM_PROMPT_RAG = (
    "You are a careful question-answering assistant.\n"
    "You answer strictly from the CONTEXT provided in the user message.\n"
    "\n"
    "Rules:\n"
    "1. Use only facts found in CONTEXT. Do not use outside knowledge.\n"
    "2. If CONTEXT does not contain the answer, reply exactly:\n"
    '   "The provided context does not contain the answer."\n'
    "3. Cite the doc_name (and page when useful) for every fact you state.\n"
    "4. Keep answers concise — one or two sentences unless the question demands more."
)

NO_CONTEXT_BLOCK = "No relevant context was retrieved."

CHUNK_SEPARATOR = "\n\n---------\n\n"

CLOSING_INSTRUCTION = (
    "Answer using only the sources above. Cite the doc_name for each fact.\n"
    "If the answer is not in the sources, say so explicitly."
)


def answer_with_rag(query: str, k: int = 4) -> dict:
    hits = vectorstore.similarity_search(query, k=k)

    if not hits:
        context_block = NO_CONTEXT_BLOCK
    else:
        blocks = []
        for i, h in enumerate(hits, 1):
            m = h.metadata
            header = f"[Source {i} | doc: {m.get('doc_name')} | page: {m.get('page_number')}]"
            blocks.append(f"{header}\n{h.page_content}")
        context_block = CHUNK_SEPARATOR.join(blocks)

    user_message = (
        f"CONTEXT:\n{context_block}\n\n"
        f"QUESTION: {query}\n\n"
        f"{CLOSING_INSTRUCTION}"
    )

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_RAG},
            {"role": "user", "content": user_message},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    answer = resp.choices[0].message.content.strip()

    retrieved_chunks = [
        {
            "doc_name": h.metadata.get("doc_name"),
            "page_number": h.metadata.get("page_number"),
            "content": h.page_content,
        }
        for h in hits
    ]

    return {"answer": answer, "retrieved_chunks": retrieved_chunks}


In [16]:
# ---- Task 4: sanity check on a probe question ----
probe_q = ds.loc[ds["financebench_id"] == "financebench_id_00005", "question"].iloc[0]
result = answer_with_rag(probe_q, k=4)

print("Q:", probe_q)
print()
print("ANSWER:")
print(result["answer"])
print()
print("RETRIEVED CHUNKS:")
for i, c in enumerate(result["retrieved_chunks"], 1):
    snippet = c["content"][:140].replace("\n", " ")
    print(f"  [{i}] doc={c['doc_name']} page={c['page_number']}")
    print(f"      {snippet}")


Q: Does Corning have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.

ANSWER:
The provided context does not contain the answer. 

The Corning documents (CORNING_2022_10K) do not mention working capital, while the 3M documents (3M_2022_10K and 3M_2023Q2_10Q) discuss working capital, but are related to 3M, not Corning (doc: 3M_2022_10K, page: 37 and doc: 3M_2023Q2_10Q, page: 70).

RETRIEVED CHUNKS:
  [1] doc=CORNING_2022_10K page=101
      (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Gro
  [2] doc=CORNING_2022_10K page=102
      (1) Corning obtained a controlling interest in HSG during the third quarter of 2020 and has consolidated results in Hemlock and Emerging Gro
  [3] doc=3M_2022_10K page=37
      not defined under U.S. generally accepted accounting principles and may not be computed

## Task 5 - run and compare (10 points)
Run the same 10 questions from Task 1 through your RAG pipeline. For each question, put
the naive answer and the RAG answer side by side with the ground truth, and comment on:
1. Did RAG help? Cases where the naive model refused or hallucinated, and RAG
produced a grounded answer.
2. Did RAG hurt? Cases where the naive model happened to be right (from
memorization) and RAG made it worse - e.g., retrieved the wrong filing, or pulled
chunks that confused the model.
3. Patterns by question_type - does RAG help more on domain-relevant than on
novel-generated, or vice versa? Any hypothesis why?

Deliverables:
1. The code.
2. An xlsx file with columns: financebench_id | question_type | question |
naive_answer | RAG_answer | ground_truth
Name the file assignment2_run_and_compare.
3. A discussion in a markdown cell in the notebook addressing the three points above.

In [19]:
# ---- Task 5: run the same 10 questions through RAG and compare ----
rag_answers = []
for i, row in sample.iterrows():
    print(f"[{i+1}/{len(sample)}] {row['financebench_id']} ({row['question_type']})")
    try:
        rag_answers.append(answer_with_rag(row["question"], k=4)["answer"])
    except Exception as e:
        rag_answers.append(f"ERROR: {e}")
    time.sleep(0.4)

compare = (
    sample[["financebench_id", "question_type", "question", "naive_answer", "answer"]]
      .rename(columns={"answer": "ground_truth"})
)
compare.insert(4, "RAG_answer", rag_answers)
compare = compare[["financebench_id", "question_type", "question",
                   "naive_answer", "RAG_answer", "ground_truth"]]

compare.to_excel("assignment2_run_and_compare.xlsx", index=False)
compare


[1/10] financebench_id_00005 (domain-relevant)
[2/10] financebench_id_00070 (domain-relevant)
[3/10] financebench_id_00080 (domain-relevant)
[4/10] financebench_id_00206 (domain-relevant)
[5/10] financebench_id_00215 (domain-relevant)
[6/10] financebench_id_00283 (novel-generated)
[7/10] financebench_id_00288 (novel-generated)
[8/10] financebench_id_00299 (novel-generated)
[9/10] financebench_id_00302 (novel-generated)
[10/10] financebench_id_00382 (novel-generated)


,financebench_id,question_type,question,naive_answer,RAG_answer,ground_truth
0,financebench_id_00005,domain-relevant,Does Corning have positive working capital bas...,To determine if Corning has positive working c...,The provided context does not contain the answ...,Yes. Corning had a positive working capital am...
1,financebench_id_00070,domain-relevant,Does American Water Works have positive workin...,To determine whether American Water Works has ...,The provided context does not contain the answer.,"No, American Water Works had negative working ..."
2,financebench_id_00080,domain-relevant,Does Paypal have positive working capital base...,To determine if PayPal has positive working ca...,The provided context does not contain the answer.,Yes. Paypal has a positive working capital of ...
3,financebench_id_00206,domain-relevant,Are JPM's gross margins historically consisten...,"To answer this question, I'll need to look at ...",The provided context does not contain the answer.,"Since JPM is a financial institution, gross ma..."
4,financebench_id_00215,domain-relevant,Is Verizon a capital intensive business based ...,To determine if Verizon is a capital-intensive...,"Yes, Verizon appears to be a capital-intensive...",Yes. Verizon's capital intensity ratio was app...
5,financebench_id_00283,novel-generated,How much does Pfizer expect to pay to spin off...,Pfizer announced plans to spin off its Upjohn ...,"Pfizer expects to pay $1.6 billion, primarily ...",77.78
6,financebench_id_00288,novel-generated,Was there any drop in Cash & Cash equivalents ...,I don't have enough information to answer your...,"Yes, there was a drop in Cash & Cash equivalen...","Yes, there was a decline of ~42% between FY202..."
7,financebench_id_00299,novel-generated,Which of JPM's business segments had the lowes...,According to JPMorgan Chase's Q1 2021 earnings...,"According to doc: JPMORGAN_2022Q2_10Q, the net...",Corporate. Its net revenue was -$473 million.
8,financebench_id_00302,novel-generated,Did Pfizer grow its PPNE between FY20 and FY21?,To answer whether Pfizer grew its Pre-tax Prof...,The provided context does not contain the answer.,"Yes, change in PPNE was positive year over year"
9,financebench_id_00382,novel-generated,Which region had the Highest EBITDAR Contribut...,I don't have access to real-time data or MGM's...,"According to the sources, Las Vegas Strip Reso...",Las Vegas resorts contributed ~90% of company ...
